# Measure your chip's roofline

*others · beginner* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/others/chip_roofline)

How fast is the machine this runs on? Time one large memory copy to get the bandwidth $\beta$, and an $n\times n$ matrix product $C_{ij} = A_{ik}B_{kj}$ in FP64, FP32, TF32, BF16 and FP16 to get each precision's rate $\pi$; the roofline $P(I) = \min(\pi, \beta I)$ then says which kernels your chip runs at full speed and which wait for memory. Coding Skills ▸ Chips places your result next to the spec sheets of other chips.

**Vector form**

$$P(I) = \min(\pi,\ \beta\, I),\qquad I_\text{ridge} = \pi / \beta$$

**Index (tensor) form**

$$C_{ij} = A_{ik}\,B_{kj}:\quad 2n^3\ \text{FLOP over}\ 3n^2 b\ \text{bytes},\qquad I = \frac{2n}{3b}$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $\beta$ | `measure_bandwidth(copy_mb)['gbs']` | () | memory bandwidth from a large copy (read + write), GB/s |
| $\pi$ | `measure_matmul(p, n)['tflops']` | () | achieved matrix-product rate in precision $p$, TFLOP/s |
| $C_{ij} = A_{ik}B_{kj}$ | `a @ b` | (n, n) | repeated $k$ ⇒ summed: $n^3$ multiply-adds |
| $I$ | `intensity(n, b)` | () | FLOP per byte moved: $2n/(3b)$ for $b$-byte elements |
| $P(I)$ | `roofline(pi, beta, I)` | () | the most a kernel of intensity $I$ can reach |

Spec sheets give peaks; a measurement gives what PyTorch actually gets on your machine, usually 50–90% of the peak for a large matrix product. GPU work is asynchronous, so every timing waits for the device to finish (synchronize) and takes the median of several runs after a warm-up. On NVIDIA GPUs the FP32 product is measured twice: once in true FP32 and once with TF32 tensor cores (torch.set_float32_matmul_precision("high")), the setting that silently trades mantissa bits for speed. Apple's GPU backend (MPS) has no float64, so FP64 is skipped there; consumer GPUs run FP64 at a small fraction of their FP32 rate, which matters for physics. Try a bigger matrix size: the rate climbs until the product is compute-bound, the memory copy stays at the bandwidth.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/others/chip_roofline/main.py
"""Main loop: measure the memory bandwidth and the matmul rate in each precision, print them, draw the roofline.

The last printed line is `CHIP_RESULT {...}` (JSON): the Coding Skills ▸ Chips tab reads it to place this machine
on its roofline and in its comparison table.
"""

import json
from pathlib import Path

from problems._common.runtime import cli, device, fast_mode

from . import plot, solve, tensors


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    fast = fast_mode()
    n = 256 if fast else int(params["matrix_size"])
    mb = 8 if fast else int(params["copy_mb"])
    repeats = 2 if fast else int(params["repeats"])

    result = {"device": solve.device_name(dev), "deviceType": dev.type, "n": n}
    result["bandwidth"] = solve.measure_bandwidth(mb, dev, repeats)
    print(f"device: {result['device']} ({dev.type})")
    print(f"memory copy of {mb} MB: {result['bandwidth']['gbs']:.1f} GB/s")
    result["matmul"] = []
    for name in tensors.PRECISIONS:
        m = solve.measure_matmul(name, n, dev, repeats)
        result["matmul"].append(m)
        if "skipped" in m:
            print(f"{name.upper():5s} skipped: {m['skipped']}")
        else:
            note = f"  ({m['slow']})" if "slow" in m else ""
            rate = f"{m['tflops']:.3f} TFLOP/s  ({m['seconds'] * 1e3:.2f} ms)"
            print(f"{name.upper():5s} {m['n']}x{m['n']} matmul: {rate}{note}")
    print("CHIP_RESULT " + json.dumps(result, separators=(",", ":")))
    return plot.roofline(result, out_dir / "chip_roofline.png")


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/others/chip_roofline/equation.py
"""The roofline model: how fast a kernel can go is the smaller of the chip's compute peak and what its memory can feed.

    P(I) = min(π, β · I)

with π the peak arithmetic rate (FLOP/s), β the memory bandwidth (byte/s) and I the kernel's arithmetic intensity
(FLOP per byte moved). A matrix product C_ij = A_ik B_kj of n×n matrices does 2n³ FLOP (n³ multiply-adds) and, at
best, moves the three matrices once: 3 n² b bytes for b-byte elements, so I = 2n / (3b).
"""


def matmul_flops(n: int) -> float:
    """FLOP of an n×n by n×n matrix product: n³ multiply-adds, 2 FLOP each."""
    return 2.0 * n**3


def matmul_bytes(n: int, elem_bytes: int) -> float:
    """Bytes of A, B and C, each read or written once (the best case a tiled kernel approaches)."""
    return 3.0 * n * n * elem_bytes


def intensity(n: int, elem_bytes: int) -> float:
    """Arithmetic intensity of the matrix product, FLOP per byte: 2n / (3b)."""
    return matmul_flops(n) / matmul_bytes(n, elem_bytes)


def copy_bytes(numel: int, elem_bytes: int) -> float:
    """A copy dst ← src reads every element once and writes it once."""
    return 2.0 * numel * elem_bytes


def roofline(peak_flops: float, bandwidth: float, intensity_: float) -> float:
    """Attainable FLOP/s at a given arithmetic intensity."""
    return min(peak_flops, bandwidth * intensity_)


def ridge(peak_flops: float, bandwidth: float) -> float:
    """The intensity (FLOP/byte) where the slope meets the flat ceiling: below it a kernel waits for memory."""
    return peak_flops / bandwidth


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/others/chip_roofline/tensors.py
"""The operands: square matrices in each precision, and a large buffer for the bandwidth test."""

import torch

# name -> (dtype, matmul precision for float32 on NVIDIA: "highest" = true FP32, "high" = TF32 tensor cores)
PRECISIONS: dict[str, tuple[torch.dtype, str | None]] = {
    "fp64": (torch.float64, None),
    "fp32": (torch.float32, "highest"),
    "tf32": (torch.float32, "high"),
    "bf16": (torch.bfloat16, None),
    "fp16": (torch.float16, None),
}


def supported(name: str, device: torch.device) -> tuple[bool, str]:
    """Whether a precision can run on the device, and why not."""
    dtype, mode = PRECISIONS[name]
    if device.type == "mps" and dtype == torch.float64:
        return False, "Apple's GPU backend (MPS) has no float64"
    if mode == "high" and device.type != "cuda":
        return False, "TF32 is an NVIDIA tensor-core mode"
    return True, ""


def operands(n: int, dtype: torch.dtype, device: torch.device) -> tuple[torch.Tensor, torch.Tensor]:
    """Two random n×n matrices (values in [-1, 1), fixed seed)."""
    g = torch.Generator().manual_seed(0)
    a = (torch.rand(n, n, generator=g) * 2 - 1).to(device=device, dtype=dtype)
    b = (torch.rand(n, n, generator=g) * 2 - 1).to(device=device, dtype=dtype)
    return a, b


def buffers(megabytes: int, device: torch.device) -> tuple[torch.Tensor, torch.Tensor]:
    """Source and destination float32 buffers of the given size, for the copy (bandwidth) test."""
    numel = megabytes * 1024 * 1024 // 4
    return torch.ones(numel, device=device), torch.empty(numel, device=device)


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/others/chip_roofline/solve.py
"""Timing on any device: synchronise (GPU work is asynchronous), warm up, then take the median of several runs."""

import statistics
import time
from collections.abc import Callable

import torch

from . import equation, tensors


def sync(device: torch.device) -> None:
    if device.type == "cuda":
        torch.cuda.synchronize(device)
    elif device.type == "mps":
        torch.mps.synchronize()
    elif device.type == "xpu":
        torch.xpu.synchronize(device)


def median_seconds(fn: Callable[[], object], device: torch.device, repeats: int, warmup: int = 2) -> float:
    for _ in range(warmup):
        fn()
    sync(device)
    times = []
    for _ in range(repeats):
        t0 = time.perf_counter()
        fn()
        sync(device)
        times.append(time.perf_counter() - t0)
    return statistics.median(times)


def measure_matmul(name: str, n: int, device: torch.device, repeats: int) -> dict:
    """Achieved FLOP/s of an n×n matrix product in one precision (or why it was skipped)."""
    ok, why = tensors.supported(name, device)
    if not ok:
        return {"precision": name, "skipped": why}
    dtype, mode = tensors.PRECISIONS[name]
    # CPUs without a BF16/FP16 matrix unit (AMX, AVX-512 BF16/FP16, Arm SME) emulate these formats very slowly:
    # a small probe first, and the full size only when the format is not far slower than FP32
    if device.type == "cpu" and dtype in (torch.bfloat16, torch.float16) and n > 256:
        probe, base = (measure_matmul(p, 256, device, 2) for p in (name, "fp32"))
        if probe["tflops"] < 0.1 * base["tflops"]:
            return {**probe, "slow": f"no fast {name.upper()} on this CPU: measured at n = 256 only"}
    before = torch.get_float32_matmul_precision()
    if mode:
        torch.set_float32_matmul_precision(mode)
    try:
        a, b = tensors.operands(n, dtype, device)
        s = median_seconds(lambda: a @ b, device, repeats)
    finally:
        torch.set_float32_matmul_precision(before)
    flops = equation.matmul_flops(n) / s
    return {
        "precision": name,
        "n": n,
        "seconds": s,
        "tflops": flops / 1e12,
        "intensity": equation.intensity(n, a.element_size()),
    }


def measure_bandwidth(megabytes: int, device: torch.device, repeats: int) -> dict:
    """Achieved memory bandwidth of a large copy, in GB/s (10^9 bytes per second)."""
    src, dst = tensors.buffers(megabytes, device)
    s = median_seconds(lambda: dst.copy_(src), device, repeats)
    return {"megabytes": megabytes, "seconds": s, "gbs": equation.copy_bytes(src.numel(), src.element_size()) / s / 1e9}


def cpu_name() -> str:
    """The processor's marketing name (e.g. "Apple M5 Pro", "AMD Ryzen 9 7950X"), else the architecture."""
    import platform
    import subprocess
    from pathlib import Path

    try:
        if platform.system() == "Darwin":
            return subprocess.run(
                ["sysctl", "-n", "machdep.cpu.brand_string"], capture_output=True, text=True, timeout=5
            ).stdout.strip()
        cpuinfo = Path("/proc/cpuinfo")
        if cpuinfo.exists():
            for line in cpuinfo.read_text().splitlines():
                if line.lower().startswith(("model name", "hardware", "model")) and ":" in line:
                    return line.split(":", 1)[1].strip()
    except (OSError, subprocess.SubprocessError):
        pass
    return platform.processor() or platform.machine() or "CPU"


def device_name(device: torch.device) -> str:
    if device.type == "cuda":
        return torch.cuda.get_device_name(device)
    if device.type == "mps":
        return f"{cpu_name()} GPU (MPS)"
    return f"{cpu_name()} CPU"


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/others/chip_roofline/plot.py
"""The measured roofline: the bandwidth slope, each precision's measured matmul rate, and the ridge points."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from problems._common.plotting import save_png

from . import equation

COLOURS = {"fp64": "#7c3aed", "fp32": "#2563eb", "tf32": "#0891b2", "bf16": "#16a34a", "fp16": "#f59e0b"}


def roofline(result: dict, path: Path) -> Path:
    bw = result["bandwidth"]["gbs"] * 1e9  # byte/s
    runs = [m for m in result["matmul"] if "tflops" in m]
    fig, ax = plt.subplots(figsize=(7.2, 4.4))
    xs = np.logspace(-2, 4, 200)
    top = max([m["tflops"] for m in runs] + [1e-3]) * 1e12
    ax.plot(xs, np.minimum(bw * xs, top) / 1e12, color="#9ca3af", lw=1.2, ls="--", label=f"memory: {bw / 1e9:.0f} GB/s")
    for m in runs:
        c = COLOURS.get(m["precision"], "#374151")
        peak = m["tflops"] * 1e12
        ax.plot(xs, [equation.roofline(peak, bw, x) / 1e12 for x in xs], color=c, lw=1.6)
        ax.scatter([m["intensity"]], [m["tflops"]], color=c, s=36, zorder=3)
        ax.annotate(
            f"{m['precision'].upper()} {m['tflops']:.2f} TFLOP/s",
            (m["intensity"], m["tflops"]),
            textcoords="offset points",
            xytext=(6, 4),
            fontsize=8,
            color=c,
        )
        r = equation.ridge(peak, bw)
        ax.axvline(r, color=c, lw=0.6, ls=":")
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel("arithmetic intensity (FLOP / byte)")
    ax.set_ylabel("TFLOP/s")
    ax.set_title(f"Measured roofline: {result['device']}", fontsize=10)
    ax.grid(True, which="both", lw=0.3, alpha=0.5)
    ax.legend(loc="lower right", fontsize=8)
    return save_png(fig, path)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'matrix_size': 4096,
    'copy_mb': 512,
    'repeats': 5
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.others.chip_roofline")]:
    del sys.modules[name]
main = importlib.import_module("problems.others.chip_roofline.main")
problem_dir = Path("problems/others/chip_roofline")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
